In [1]:
%%writefile semantico.py
import copy

#--------------Analisis Semantico------------------------
class AnalizadorSemantico:
    def __init__(self):
        self.tabla_simbolos = TablaSimbolos()

    def analizar(self, nodo):
        if isinstance(nodo, NodoPrograma):
            for funcion in nodo.funciones:
                self.analizar(funcion)
            self.analizar(main)
        elif isinstance(nodo, NodoFuncion):
            self.tabla_simbolos.declarar_funcion(nodo.nombre, nodo.tipo, nodo.parametros)
            for instruccion in nodo.cuerpo:
                self.analizar(instruccion)
        elif isinstance(nodo, NodoAsignacion):
            tipo_expr = self.analizar(nodo.expresion)
            if tipo_expr != nodo.tipo:
                raise Exception(f"Error: no conciden los tipos {nodo.tipo} != {tipo_expr}")

            self.tabla_simbolos.declarar_variable(nodo.nombre, nodo.tipo)
        elif isinstance(nodo, NodoOperacion):
            tipo_izq = self.analizar(nodo.izquierda)
            tipo_der = self.analizar(nodo.derecha)
            if tipo_izq != tipo_der:
                raise Exception(f"Error: Tipos Incompatibles en la expresion {tipo_izq} {nodo.operador} {tipo_der}")

    def visitar_NodoFuncion(self, nodo):
        if nodo.nombre[1] in self.tabla_simbolos:
            raise Exception((f'Error Semantico: la funcion {nodo.nombre[1]} ya esta definida'))
        self.tabla_simbolos[nodo.nombre[1]] = {'tipo': nodo.parametros[0].tipo[1], 'parametro': nodo.parametros}
        for param in nodo.parametros:
            self.tabla_simbolos[param.nombre[1]] = {'tipo': param.tipo[1]}

        for instruccion in nodo.cuerpo:
            self.analizar(instruccion)

    def visitar_NodoAsignacion(self, nodo):
        tipo_expresion = self.analizar(nodo.expresion)
        self.tabla_simbolos[nodo.nombre[1]] = {'tipo':tipo_expresion}

    def visitar_NodoOperacion(self, nodo):
        tipo_izquierda = self.analizar(nodo.izquierda)
        tipo_derecha = self.analizar(nodo.derecha)

        if tipo_izquierda != tipo_derecha:
            raise Exception(f'Error semantico: Operacion entre tipos incompatibles ({tipo_izquierda} y {tipo_derecha})')

        return tipo_izquierda

    def visitar_NodoNumero(self, nodo):
        return 'int' if '.' not in nodo.valor[1] else 'float'

    def visitar_NodoIdentificador(self, nodo):
        if nodo.nombre[1] not in self.tabla_simbolos:
            raise Exception(f'Error semantica: La variable {nodo.nombre[1]} no esta definida')
        return self.tabla_simbolos[nodo.nobmre[1]]['tipo']

    def visitar_NodoRetorno(self, nodo):
        return self.analizar(nodo.expresion)
    
    def visitar_NodoPrograma(self, nodo):
        for funcion in nodo.funciones:
            self.analizar(funcion)
        self.analizar(nodo.main)


class TablaSimbolos:
    def __init__(self):
        self.variables = {}    #Almacena variables {nombre:tipo}
        self.funciones = {}    # Almacena funciones {nombre:(tipo_ret, [parametros])}

    def declarar_variables(self, nombre, tipo):
        if nombre in self.variables:
            raise Exception(f"Error: Variable '{nombre}' ya declarada")
        self.variables[nombre] = tipo

    def obtener_tipo_variable(self, nombre):
        if nombre not in self.variables:
            raise Exception(f"Error: Variabl;e '{nombre}' no definida")
        return self.variables[nombre]

    def declarar_funcion(self, nombre, tipo_retorno, parametros):
        if nombre in self.funciones:
            raise Exception(f"Error: Funcion '{nombre}' ya definida")
        self.funciones[nombre] = (tipo_retorno, parametros)

    def obtener_infop_funcion(self, nombre):
        if nobmre not in self.funciones:
            raise Exception(f"Error: funcion '{nombre}' no definida")
        return self.funciones[nombre]


# -------------------------------------------------------
#  TABLA DE SIMBOLOS JERARQUICA  (Pila de ambitos)
# -------------------------------------------------------
class TablaSimbolosJerarquica:
    def __init__(self):
        self.ambitos           = [{}]   # indice 0 = global
        self.funciones         = {}
        self.historial_ambitos = []     # para imprimir_resumen_final()

    def abrir_ambito(self, etiqueta="bloque"):
        self.ambitos.append({})
        self._registrar_snapshot(f"ABRIR {etiqueta}")

    def cerrar_ambito(self, etiqueta="bloque"):
        if len(self.ambitos) == 1:
            raise Exception("Error: no se puede cerrar el ambito global")
        self._registrar_snapshot(f"ANTES DE CERRAR {etiqueta}")
        self.ambitos.pop()

    def declarar_variable(self, nombre, tipo):
        ambito_actual = self.ambitos[-1]
        if nombre in ambito_actual:
            raise Exception(f"Error Semantico: '{nombre}' ya declarada en este ambito")
        ambito_actual[nombre] = tipo

    def obtener_tipo_variable(self, nombre):
        # Busca de adentro hacia afuera -> implementa shadowing
        for ambito in reversed(self.ambitos):
            if nombre in ambito:
                return ambito[nombre]
        raise Exception(f"Error Semantico: '{nombre}' no declarada en ningun ambito visible")

    def declarar_funcion(self, nombre, tipo_retorno, parametros):
        if nombre in self.funciones:
            raise Exception(f"Error Semantico: funcion '{nombre}' ya definida")
        self.funciones[nombre] = (tipo_retorno, parametros)

    def snapshot_manual(self, etiqueta):
        self._registrar_snapshot(etiqueta)

    def _registrar_snapshot(self, etiqueta):
        self.historial_ambitos.append({
            "momento":     etiqueta,
            "pila":        copy.deepcopy(self.ambitos),
            "profundidad": len(self.ambitos),
        })

    def imprimir_estado(self, titulo="Estado actual"):
        separador = "=" * 55
        print(f"\n{separador}")
        print(f"  {titulo}")
        print(separador)
        for i, ambito in enumerate(self.ambitos):
            etq = "GLOBAL" if i == 0 else f"Nivel {i}"
            print(f"  [{etq}]  {ambito if ambito else '(vacio)'}")
        print(f"  Profundidad: {len(self.ambitos)} ambito(s) activos")
        print(separador)


# -------------------------------------------------------
#  NODO BLOQUE  (bloque anonimo { })
# -------------------------------------------------------
class NodoBloque:
    def __init__(self, instrucciones):
        self.instrucciones = instrucciones


# -------------------------------------------------------
#  ANALIZADOR SEMANTICO V2  (usa TablaSimbolosJerarquica)
# -------------------------------------------------------
class AnalizadorSemanticoV2:
    def __init__(self):
        self.tabla   = TablaSimbolosJerarquica()
        self.errores = []

    def analizar(self, nodo):
        metodo = getattr(self, f"_visitar_{type(nodo).__name__}", self._nodo_desconocido)
        return metodo(nodo)

    def _nodo_desconocido(self, nodo):
        self.errores.append(f"Nodo desconocido: {type(nodo).__name__}")

    def _visitar_NodoFuncion(self, nodo):
        nombre = nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre
        tipo_r = nodo.tipo[1]   if isinstance(nodo.tipo,   tuple) else nodo.tipo
        self.tabla.declarar_funcion(nombre, tipo_r, nodo.parametros)
        self.tabla.abrir_ambito(f"funcion '{nombre}'")
        for param in nodo.parametros:
            p_nombre = param.nombre[1] if isinstance(param.nombre, tuple) else param.nombre
            p_tipo   = param.tipo[1]   if isinstance(param.tipo,   tuple) else param.tipo
            self.tabla.declarar_variable(p_nombre, p_tipo)
        for instruccion in nodo.cuerpo:
            self.analizar(instruccion)
        self.tabla.cerrar_ambito(f"funcion '{nombre}'")

    def _visitar_NodoBloque(self, nodo):
        self.tabla.abrir_ambito("bloque anonimo")
        for instruccion in nodo.instrucciones:
            self.analizar(instruccion)
        self.tabla.cerrar_ambito("bloque anonimo")

    def _visitar_NodoAsignacion(self, nodo):
        nombre = nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre
        tipo   = nodo.tipo[1]   if isinstance(nodo.tipo,   tuple) else nodo.tipo
        tipo_expr = self._tipo_expresion(nodo.expresion)
        if tipo_expr and tipo != tipo_expr:
            self.errores.append(
                f"Error Semantico [Tipo]: '{nombre}' es '{tipo}' pero la expresion es '{tipo_expr}'"
            )
        try:
            self.tabla.declarar_variable(nombre, tipo)
        except Exception as e:
            self.errores.append(str(e))

    def _tipo_expresion(self, nodo):
        from sintactico_ast import NodoNumero, NodoIdentificador, NodoOperacion
        if isinstance(nodo, NodoNumero):
            return 'float' if '.' in str(nodo.valor[1]) else 'int'
        elif isinstance(nodo, NodoIdentificador):
            nombre = nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre
            try:
                return self.tabla.obtener_tipo_variable(nombre)
            except Exception as e:
                self.errores.append(str(e))
                return None
        elif isinstance(nodo, NodoOperacion):
            ti = self._tipo_expresion(nodo.izquierda)
            td = self._tipo_expresion(nodo.derecha)
            if ti and td and ti != td:
                op = nodo.operador[1] if isinstance(nodo.operador, tuple) else nodo.operador
                self.errores.append(
                    f"Error Semantico [Tipo]: operacion '{op}' entre '{ti}' y '{td}'"
                )
                return None
            return ti
        return None

    def imprimir_errores(self):
        separador = "=" * 55
        print(f"\n{separador}")
        print("  ERRORES SEMANTICOS")
        print(separador)
        if not self.errores:
            print("  Sin errores.")
        else:
            for i, e in enumerate(self.errores, 1):
                print(f"  [{i}] {e}")
        print(separador)


# -------------------------------------------------------
#  SIMULACION DEL EJERCICIO
# -------------------------------------------------------
def ejecutar_simulacion_laberinto():
    tabla   = TablaSimbolosJerarquica()
    errores = []

    separador = "=" * 55
    print(f"\n{separador}")
    print("  SIMULACION: El Laberinto de los Ambitos")
    print(separador)

    # Ambito global
    tabla.declarar_variable("x", "int")
    tabla.declarar_funcion("test", "void", [("int", "a")])

    # Ambito funcion test
    tabla.abrir_ambito("funcion 'test'")
    tabla.declarar_variable("a", "int")
    tabla.declarar_variable("y", "int")

    # -- MOMENTO A --
    tabla.snapshot_manual("MOMENTO A: despues de 'int y = a * 2'")
    tabla.imprimir_estado("MOMENTO A -- despues de declarar 'int y = a * 2;'")

    # Ambito bloque interno { }
    tabla.abrir_ambito("bloque interno")
    tabla.declarar_variable("x", "float")          # shadowing

    tipo_y = tabla.obtener_tipo_variable("y")
    tipo_x = tabla.obtener_tipo_variable("x")
    if tipo_y != tipo_x:
        errores.append(
            f"Error Semantico [1] -- Tipos incompatibles: "
            f"'y'({tipo_y}) + 'x'({tipo_x}) en 'y = y + x'"
        )

    # -- MOMENTO B --
    tabla.snapshot_manual("MOMENTO B: antes de cerrar el bloque interno")
    tabla.imprimir_estado("MOMENTO B -- antes de cerrar el bloque interno")
    print(f"  Shadowing activo: 'x' visible = '{tipo_x}' (float local oculta al int global)")

    tabla.cerrar_ambito("bloque interno")

    # escribir(z) -> z no existe
    try:
        tabla.obtener_tipo_variable("z")
    except Exception as e:
        errores.append(f"Error Semantico [2] -- {e}")

    # -- MOMENTO C --
    tabla.snapshot_manual("MOMENTO C: al llegar a 'escribir(z)'")
    tabla.imprimir_estado("MOMENTO C -- al llegar a 'escribir(z);'")

    tabla.cerrar_ambito("funcion 'test'")

    # Reporte
    print(f"\n{separador}")
    print("  ERRORES DETECTADOS")
    print(separador)
    for e in errores:
        print(f"  - {e}")
    print(f"  - Error Semantico [3] -- Shadowing: 'float x' en el bloque interno "
          f"oculta a 'int x' global, bloqueando acceso a ella dentro del bloque.")
    print(separador)

    return tabla


def generar_c3d_bloque_interno():
    separador = "=" * 55
    print(f"\n{separador}")
    print("  CODIGO DE TRES DIRECCIONES -- Bloque interno")
    print(separador)
    lineas = [
        ("//", "Inicio bloque interno { }"),
        ("x_local",  "= 5.5",         "# float x = 5.5  (shadowing -> x_local)"),
        ("t1",       "= y + x_local",  "# y(int) + x_local(float) -> ERROR tipo"),
        ("y",        "= t1",           ""),
        ("//", "Fin bloque interno }"),
        ("//", "De vuelta en ambito de test()"),
        ("t2",       "= y + 1",        ""),
        ("x_global", "= t2",           "# x = y + 1  (x global int)"),
    ]
    for l in lineas:
        if l[0] == "//":
            print(f"  // {l[1]}")
        else:
            print(f"  {l[0]:12} {l[1]:20} {l[2]}")
    print(separador)
    print("  x_local  -> float x del bloque interno")
    print("  x_global -> int   x del ambito global")
    print("  t1, t2   -> temporales unicos")
    print(separador)


def explicar_shadowing():
    separador = "=" * 55
    print(f"\n{separador}")
    print("  FENOMENO DEL SHADOWING -- Pregunta 3")
    print(separador)
    print("""
  Pila en MOMENTO B:
    [GLOBAL]  -> { x: int  }
    [Nivel 1] -> { a: int, y: int }
    [Nivel 2] -> { x: float }        <- tope

  obtener_tipo_variable('x') recorre de adentro -> afuera:
    1. Busca en Nivel 2 -> ENCONTRADA: float
    2. Se detiene. No llega al global.

  En  y = y + x:
      y  ->  int
      x  ->  float   (la local, no la global)
      int + float   ->  ERROR SEMANTICO
""")
    print(separador)


def imprimir_resumen_final(tabla):
    esperados = {
        "MOMENTO A": [{"x": "int"}, {"a": "int", "y": "int"}],
        "MOMENTO B": [{"x": "int"}, {"a": "int", "y": "int"}, {"x": "float"}],
        "MOMENTO C": [{"x": "int"}, {"a": "int", "y": "int"}],
    }
    separador  = "=" * 55
    separador2 = "-" * 50
    print(f"\n{separador}")
    print("  RESUMEN FINAL -- Historial de Pila de Ambitos")
    print(separador)
    for snap in tabla.historial_ambitos:
        if snap["momento"].startswith("MOMENTO"):
            print(f"  {snap['momento']}")
            for i, a in enumerate(snap["pila"]):
                print(f"     [{'GLOBAL' if i==0 else f'Nivel {i}'}] {a}")
            etq = snap["momento"].split(":")[0].strip()
            ok  = snap["pila"] == esperados.get(etq, None)
            print(f"     -> {'COINCIDE' if ok else 'DIFIERE'}")
            print(f"  {separador2}")
    print(separador)

Overwriting semantico.py
